# C4 — Leakage-Safe Split

Default evaluation split: **case-grouped 70/15/15**.

Why case grouping:
- multiple decision rows can belong to one case;
- row-level random splitting would leak case history.

Customer/subscription overlap is audited separately.

Temporal holdout is evaluated as a diagnostic, not automatically chosen as the
primary split.

Learned preprocessing is fitted only on the training partition.

In [1]:

from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "ml_ready").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

SOURCE = PROJECT_ROOT / "data" / "ml_ready" / "ml_datamart_clean.csv"
SPLIT_DIR = PROJECT_ROOT / "data" / "c4_splits"
REPORT_DIR = PROJECT_ROOT / "reports" / "c4"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SOURCE, low_memory=False)
target_col = "y" if "y" in df.columns else "y_intervention"

print("Rows:", len(df))
print("Target:", target_col)


Rows: 157423
Target: y


## 1. Customer / Subscription / Case Dependence

In [2]:

rows = []

for col in ["case_id", "customer_id", "subscription_id"]:
    if col not in df.columns:
        continue

    counts = df.groupby(col).size()
    rows.append({
        "entity": col,
        "unique_entities": int(df[col].nunique()),
        "entities_with_multiple_decisions": int((counts > 1).sum()),
        "max_rows_per_entity": int(counts.max()),
        "mean_rows_per_entity": float(counts.mean()),
    })

dependence = pd.DataFrame(rows)
display(dependence)

dependence.to_csv(
    REPORT_DIR / "entity_dependence_audit.csv",
    index=False,
)


,entity,unique_entities,entities_with_multiple_decisions,max_rows_per_entity,mean_rows_per_entity
0,case_id,25775,25513,13,6.107585
1,customer_id,10924,10924,109,14.410747
2,subscription_id,11596,11596,79,13.575630


## 2. Default Case-Grouped Split

In [3]:

SEED = 20260824

if "case_id" not in df.columns:
    raise ValueError("case_id is required for the default C4 split.")

gss1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=SEED,
)

train_idx, temp_idx = next(
    gss1.split(df, groups=df["case_id"])
)

train = df.iloc[train_idx].copy()
temp = df.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=SEED + 1,
)

valid_idx, test_idx = next(
    gss2.split(temp, groups=temp["case_id"])
)

valid = temp.iloc[valid_idx].copy()
test = temp.iloc[test_idx].copy()

print(f"Train: {len(train):,}")
print(f"Validation: {len(valid):,}")
print(f"Test: {len(test):,}")


Train: 110,385
Validation: 23,471
Test: 23,567


## 3. Case Exclusivity Check

In [4]:

def overlap(a, b, col):
    return len(
        set(a[col].dropna())
        & set(b[col].dropna())
    )

case_overlap = pd.DataFrame([
    {
        "comparison": "train_validation",
        "overlap": overlap(train, valid, "case_id"),
    },
    {
        "comparison": "train_test",
        "overlap": overlap(train, test, "case_id"),
    },
    {
        "comparison": "validation_test",
        "overlap": overlap(valid, test, "case_id"),
    },
])

display(case_overlap)

assert (case_overlap["overlap"] == 0).all()
print("PASS: no case appears in multiple partitions.")


,comparison,overlap
0,train_validation,0
1,train_test,0
2,validation_test,0


PASS: no case appears in multiple partitions.


## 4. Customer / Subscription Cross-Split Dependence

In [5]:

rows = []

for col in ["customer_id", "subscription_id"]:
    if col not in df.columns:
        continue

    rows.append({
        "entity": col,
        "train_validation_overlap": overlap(train, valid, col),
        "train_test_overlap": overlap(train, test, col),
        "validation_test_overlap": overlap(valid, test, col),
    })

entity_overlap = pd.DataFrame(rows)
display(entity_overlap)

entity_overlap.to_csv(
    REPORT_DIR / "customer_subscription_overlap.csv",
    index=False,
)


,entity,train_validation_overlap,train_test_overlap,validation_test_overlap
0,customer_id,2148,2139,950
1,subscription_id,2079,2096,899


## 5. Target Distribution Across Splits

In [6]:

target_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train),
        "positive_rate": float(train[target_col].mean()),
    },
    {
        "split": "validation",
        "rows": len(valid),
        "positive_rate": float(valid[target_col].mean()),
    },
    {
        "split": "test",
        "rows": len(test),
        "positive_rate": float(test[target_col].mean()),
    },
])

display(target_summary)

target_summary.to_csv(
    REPORT_DIR / "target_split_summary.csv",
    index=False,
)


,split,rows,positive_rate
0,train,110385,0.228219
1,validation,23471,0.229943
2,test,23567,0.225909


## 6. Temporal Holdout Diagnostic

In [7]:

time_col = next(
    (
        c for c in [
            "decision_ts_h",
            "decision_ts",
            "ts_h",
            "ts",
        ]
        if c in df.columns
    ),
    None,
)

if time_col is None:
    print("No suitable decision-time column found.")
else:
    tmp = df.copy()
    tmp["_time_numeric"] = pd.to_numeric(
        tmp[time_col],
        errors="coerce",
    )

    if tmp["_time_numeric"].isna().any():
        print(f"{time_col} contains missing/non-numeric values.")
    else:
        tmp = tmp.sort_values(
            ["_time_numeric"]
            + (["case_id"] if "case_id" in tmp.columns else [])
        )

        n = len(tmp)
        n_train = int(0.70 * n)
        n_valid = int(0.85 * n)

        temporal = pd.DataFrame([
            {
                "split": "temporal_train",
                "rows": n_train,
                "min_time": tmp.iloc[:n_train]["_time_numeric"].min(),
                "max_time": tmp.iloc[:n_train]["_time_numeric"].max(),
                "positive_rate": tmp.iloc[:n_train][target_col].mean(),
            },
            {
                "split": "temporal_validation",
                "rows": n_valid - n_train,
                "min_time": tmp.iloc[n_train:n_valid]["_time_numeric"].min(),
                "max_time": tmp.iloc[n_train:n_valid]["_time_numeric"].max(),
                "positive_rate": tmp.iloc[n_train:n_valid][target_col].mean(),
            },
            {
                "split": "temporal_test",
                "rows": n - n_valid,
                "min_time": tmp.iloc[n_valid:]["_time_numeric"].min(),
                "max_time": tmp.iloc[n_valid:]["_time_numeric"].max(),
                "positive_rate": tmp.iloc[n_valid:][target_col].mean(),
            },
        ])

        display(temporal)
        temporal.to_csv(
            REPORT_DIR / "temporal_holdout_diagnostic.csv",
            index=False,
        )


No suitable decision-time column found.


## 7. Train-Only Preprocessing

In [8]:

from src.ml.preprocessing import (
    FeaturePreprocessor,
    PreprocessingConfig,
)

state_candidates = [
    "amount_inr",
    "customer_tenure_days",
    "subscription_age_days",
    "prior_cycles_seen",
    "prior_successes",
    "prior_success_rate",
    "prior_failed_cases_before",
    "prior_recovered_cases_before",
    "retries_used_before",
    "contacts_sent_before",
    "hours_since_open",
    "decision_seq",
    "hour_of_day",
    "day_of_month",
    "hours_to_next_cluster_slot",
    "salary_day_proxy_dom",
    "rail",
    "bank_name",
    "segment",
    "plan_tier",
    "value_tier",
    "first_failure_class",
    "failure_class_at_decision",
    "instrument_fixed",
    "age_band",
    "state",
    "locale_pref",
    "day_of_week",
    "in_salary_cluster",
    "amount_inr_log1p",
]

state_cols = [
    c for c in state_candidates
    if c in train.columns
]

interventions = [
    c for c in ["action_type", "channel"]
    if c in train.columns
]

config = PreprocessingConfig(
    impute_numeric=True,
    add_missing_indicators=True,
    impute_categorical=True,
    rare_grouping={},
    clip_upper_quantiles={},
    create_amount_log1p=False,
    create_salary_proximity_bin=False,
)

preprocessor = FeaturePreprocessor(config=config)

# CRITICAL: fit ONLY on train.
preprocessor.fit(train[state_cols])

X_train = preprocessor.transform(train[state_cols])
X_valid = preprocessor.transform(valid[state_cols])
X_test = preprocessor.transform(test[state_cols])

for c in interventions:
    X_train[c] = train[c].astype("string").reset_index(drop=True)
    X_valid[c] = valid[c].astype("string").reset_index(drop=True)
    X_test[c] = test[c].astype("string").reset_index(drop=True)

print("Transformed train:", X_train.shape)
print("Transformed validation:", X_valid.shape)
print("Transformed test:", X_test.shape)

preprocessor.save_parameters(
    REPORT_DIR / "train_only_preprocessing_parameters.json"
)


Transformed train: (110385, 48)
Transformed validation: (23471, 48)
Transformed test: (23567, 48)


## 8. Save C4 Artifacts

In [9]:

train.to_csv(SPLIT_DIR / "train.csv", index=False)
valid.to_csv(SPLIT_DIR / "validation.csv", index=False)
test.to_csv(SPLIT_DIR / "test.csv", index=False)

metadata = {
    "seed": SEED,
    "strategy": "GroupShuffleSplit by case_id",
    "requested_fractions": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "rows": {
        "train": len(train),
        "validation": len(valid),
        "test": len(test),
    },
    "preprocessing_fit_on": "train only",
    "temporal_holdout_primary": False,
}

with open(
    REPORT_DIR / "split_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("C4 artifacts saved.")


C4 artifacts saved.


## 9. C4 Decision

**Default:** case-grouped split by `case_id`.

Customer/subscription dependence is measured separately.

A temporal holdout is retained as a diagnostic/alternative evaluation strategy
and is NOT silently substituted for the primary case-grouped split.

The next phase is **C5 — baseline model training**.